# C2: geometric-uncertainty splatting, post-hoc test (Colab, T4 is enough)

**No training.** Loads our trained uwsplat models (`gs/mvs_lpips/ckpt.pt` on Drive), adds to every Gaussian the uncertainty of its *position* given the training views that see it, re-renders the test views, and scores them. About 10 minutes for setup (most of it gsplat compiling once), then a few minutes per scene.

**Method** (`uwsplat/eval/uncertainty.py`). For Gaussian *i*, every training view *k* that sees it contributes Fisher information through its 2×3 projection Jacobian *J_ik*:

  F_i = Σ_k w_ik J_ikᵀ J_ik / σ_px² + Λ0_i,  w_ik = min(1, W_ik / W0),  P_i = F_i⁻¹

*W_ik* is the blending weight the rasteriser actually gives Gaussian *i* in view *k* (summed over its pixels). Λ0 is a weak range prior along the mean ray. At render time, for any camera with Jacobian *J*:

| variant | covariance | opacity |
|---|---|---|
| `base` | Σ (trained model) | α |
| `c2` | Σ + κP | α·√(det(JΣJᵀ+0.3) / det(JΣ′Jᵀ+0.3)) (keeps each splat's energy in that view) |
| `fade` | Σ | the c2 factor, with no blur (ablation: is the spread needed?) |
| `c2_det3` | Σ + κP | α·√(det Σ / det Σ′) (3D factor, as Mip-Splatting / Robust GS) |
| `mip` | Σ + σ_px²(min_k z/f)² I | 3D factor (Mip-Splatting's 3D filter applied after training) |
| `cull` | Σ | α, minus the Gaussians no training view sees (control: c2 and fade also drop those, since their prior-only P makes the factor tiny) |

Proven and checked in the CPU test (cell 4): in every training view that fully sees a Gaussian, J P Jᵀ ≼ σ_px² I, so training views barely move. Mip's filter is the special case that keeps only the best view and makes P isotropic.

**Pre-registered settings (do not tune on test views):** σ_px² = 0.2 px² (Mip's constant), κ = 1, W0 = 1 px, τ = 0.05, σ_z = 0.2.

**Pre-registered kill rule** (from *Novel contributions beyond RUSplatting*, adapted to one seed), per dataset:
- **(i)** On the first test view (frame 0, extrapolated), in ≥ 3 of 4 scenes: c2 raises never-seen-pixel PSNR by ≥ +1 dB, changes seen-pixel PSNR by ≥ −0.1 dB, and raises all-pixel PSNR. Interior test views: mean change within ±0.05 dB.
- **(ii)** Training-view PSNR loss of c2 is no worse than post-hoc `mip`'s (0.02 dB tolerance).
- **(iii)** c2 beats `fade` on first-view never-seen PSNR by > 0.1 dB in ≥ 3 of 4 scenes (0.1 dB stands in for the seed SD: one seed only). If fade matches, the blur adds nothing.
- **(iv)** c2 beats `cull` on first-view never-seen PSNR by > 0.1 dB in ≥ 3 of 4 scenes. If cull matches, the gain is deleting never-seen Gaussians, not the covariance. (Added before any real run.)
- A scene whose first view has no never-seen pixels counts as **not passing** (i), (iii) and (iv).

**Caveats.** These are uwsplat checkpoints, not RUSplatting ones (those are on the A40). uwsplat's first views are 94–100 % "seen" by our mask, so the effect is expected to be small. The mask projects rendered depth into training views and ignores occlusion, so it calls slightly too much "seen". Test views use the identity colour correction (as in `metrics.json`). So the all-view numbers are our identity numbers (≈ 23.7 dB on Submerged3D), not the 24.76 `neighbors` protocol.

**Inputs on Drive:** `MyDrive/uwsplat_colab/outputs_v2/<scene>/` (Submerged3D) and `outputs_seathru/<scene>/` (SeaThru-NeRF), each with `sfm/scene.json`, `images/`, `gs/mvs_lpips/ckpt.pt`.
**Outputs on Drive:** `MyDrive/uwsplat_colab/uncertainty_splat/<dataset>/<scene>/uncertainty_splat.json`, montages (GT | base | c2 | fade | c2_det3 | mip | cull), and `summary.md`. Re-running skips finished scenes.

In [ ]:
# ---- 1. settings ----
DRIVE_DIR = "/content/drive/MyDrive/uwsplat_colab"
SETS = {"Submerged3D": f"{DRIVE_DIR}/outputs_v2", "SeaThru-NeRF": f"{DRIVE_DIR}/outputs_seathru"}
SCENES = None                 # None = every scene with gs/<TAG>/ckpt.pt; or a list, e.g. ["Tokai", "Cormoran"]
TAG = "mvs_lpips"
VARIANTS = "base,c2,fade,c2_det3,mip,cull"

# pre-registered; changing these after seeing test numbers voids the kill rule
SIGMA2, KAPPA, W0, TAU, SIGMA_Z = 0.2, 1.0, 1.0, 0.05, 0.2

SAVE_IMAGES = True            # montages GT | variants per test view
FORCE = False                 # True = redo scenes whose result is already on Drive
TRAIN_IF_MISSING = False      # True = train TAG where Drive has no checkpoint (a full training run per scene: slow)

RES_DRIVE = f"{DRIVE_DIR}/uncertainty_splat"
REPO, WORK = "/content/splat_dense-sparse", "/content/c2_scenes"

import os, subprocess, torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU (T4 is fine)"
cc = torch.cuda.get_device_capability()
ARCH = f"{cc[0]}.{cc[1]}"
os.environ["TORCH_CUDA_ARCH_LIST"] = ARCH      # gsplat compiles for this GPU only (faster build)
print("torch", torch.__version__, "CUDA", torch.version.cuda, "arch", ARCH)

In [ ]:
# ---- 2. Drive ----
from google.colab import drive
drive.mount("/content/drive", timeout_ms=90000)
for dset, root in SETS.items():
    print(f"{dset:13s} {root}:", sorted(os.listdir(root)) if os.path.isdir(root) else "NOT FOUND")
os.makedirs(RES_DRIVE, exist_ok=True)

In [ ]:
%%bash -s "$REPO"
# ---- 3. code (branch dense-v2) and packages; torch is Colab's own ----
set -e
if [ -d "$1/.git" ]; then git -C "$1" fetch -q origin dense-v2 && git -C "$1" reset -q --hard origin/dense-v2
else git clone -q --branch dense-v2 https://github.com/noUpKey/splat_dense-sparse "$1"; fi
git -C "$1" log --oneline -1
test -f "$1/uwsplat/eval/uncertainty.py" || { echo "uncertainty.py missing: wrong branch?"; exit 1; }
pip -q install "lpips==0.1.4" safetensors ninja "gsplat==1.5.3"

In [ ]:
# ---- 3b. compile gsplat once (5-10 min on the first run of a fresh runtime; cached afterwards) ----
import time
t0 = time.time()
from gsplat import rasterization
dev = "cuda"
m = torch.tensor([[0.0, 0.0, 3.0]], device=dev)
q = torch.tensor([[1.0, 0.0, 0.0, 0.0]], device=dev)
s = torch.full((1, 3), 0.1, device=dev)
o = torch.tensor([0.9], device=dev)
c = torch.ones(1, 1, 3, device=dev)
K = torch.tensor([[[50.0, 0, 32], [0, 50.0, 32], [0, 0, 1]]], device=dev)
img, alpha, _ = rasterization(m, q, s, o, c, torch.eye(4, device=dev)[None], K, 64, 64, render_mode="RGB+ED")
print(f"gsplat OK ({time.time() - t0:.0f} s), peak alpha {alpha.max().item():.3f}")

In [ ]:
# ---- 4. CPU self-test of the maths (about a minute): covariance round trip, energy conservation,
#         weights = rendered alpha, training-view bound J P J^T <= sigma2 I, single-view long axis along the ray,
#         and a full run on a toy scene. Stop here if it fails.
r = subprocess.run(["python", "-m", "tests.test_uncertainty"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, "self-test failed"

In [ ]:
# ---- 5. scenes on Drive ----
import glob
jobs = []
for dset, root in SETS.items():
    if not os.path.isdir(root):
        continue
    for sd in sorted(glob.glob(f"{root}/*/")):
        sd = sd.rstrip("/")
        scene = os.path.basename(sd)
        if SCENES and scene not in SCENES:
            continue
        if not os.path.exists(f"{sd}/sfm/scene.json"):
            continue
        has = os.path.exists(f"{sd}/gs/{TAG}/ckpt.pt")
        done = os.path.exists(f"{RES_DRIVE}/{dset}/{scene}/uncertainty_splat.json")
        jobs.append(dict(dset=dset, scene=scene, src=sd, has_ckpt=has, done=done))
        size = os.path.getsize(f"{sd}/gs/{TAG}/ckpt.pt") / 2**20 if has else 0
        print(f"{dset:13s} {scene:18s} checkpoint: {'%.0f MB' % size if has else 'MISSING'}"
              f"{'   (result on Drive: skip)' if done and not FORCE else ''}")
if not any(j["has_ckpt"] for j in jobs):
    print(f"\nNo {TAG} checkpoint found. Set TRAIN_IF_MISSING = True (slow) or copy gs/{TAG}/ckpt.pt to Drive.")

In [ ]:
# ---- 6. run: P_i from the training views, re-render test + training views per variant, score ----
import json, shutil, sys

def sh(cmd):
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(cmd)}")

def copy_dir(src, dst, ignore=None):
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)

failed = []
for j in jobs:
    dst = f"{RES_DRIVE}/{j['dset']}/{j['scene']}"
    if j["done"] and not FORCE:
        continue
    local = f"{WORK}/{j['dset']}/{j['scene']}"
    print(f"\n===== {j['dset']} / {j['scene']}")
    try:
        if j["has_ckpt"]:
            for sub in ("sfm", "images"):
                copy_dir(f"{j['src']}/{sub}", f"{local}/{sub}")
            os.makedirs(f"{local}/gs/{TAG}", exist_ok=True)
            shutil.copy2(f"{j['src']}/gs/{TAG}/ckpt.pt", f"{local}/gs/{TAG}/ckpt.pt")
            if os.path.exists(f"{j['src']}/eval/{TAG}/metrics.json"):
                os.makedirs(f"{local}/eval/{TAG}", exist_ok=True)
                shutil.copy2(f"{j['src']}/eval/{TAG}/metrics.json", f"{local}/eval/{TAG}/metrics.json")
        elif TRAIN_IF_MISSING:
            copy_dir(j["src"], local, ignore=shutil.ignore_patterns("gs", "eval"))
            sh([sys.executable, "scripts/run_scene.py", "--scene", local, "--out", local,
                "--stages", "train,eval", "--variants", TAG])
            for sub in (f"gs/{TAG}", f"eval/{TAG}"):
                copy_dir(f"{local}/{sub}", f"{j['src']}/{sub}")        # keep the new checkpoint on Drive
        else:
            print("no checkpoint: skipped")
            continue
        cmd = [sys.executable, "scripts/uncertainty_splat.py", "--scene_out", local, "--tag", TAG,
               "--variants", VARIANTS, "--sigma2", str(SIGMA2), "--kappa", str(KAPPA), "--w0", str(W0),
               "--tau", str(TAU), "--sigma_z", str(SIGMA_Z)] + (["--save_images"] if SAVE_IMAGES else [])
        sh(cmd)
        ev = f"{local}/eval/{TAG}"
        res = json.load(open(f"{ev}/uncertainty_splat.json"))
        # sanity: 'base' must reproduce the scene's own evaluation (same renderer, identity colour)
        if os.path.exists(f"{ev}/metrics.json"):
            ref = json.load(open(f"{ev}/metrics.json"))["mean"]["psnr"]
            got = res["summary"]["all"]["base"]["psnr"]
            flag = "OK" if abs(got - ref) < 0.05 else "MISMATCH: check before trusting the deltas"
            print(f"sanity: base {got:.3f} dB vs metrics.json {ref:.3f} dB -> {flag}")
        os.makedirs(dst, exist_ok=True)
        shutil.copy2(f"{ev}/uncertainty_splat.json", f"{dst}/uncertainty_splat.json")
        copy_dir(f"{ev}/uncertainty", f"{dst}/montages")
        shutil.rmtree(local, ignore_errors=True)                     # keep the Colab disk small
    except Exception as e:
        print("ERROR:", e)
        failed.append(f"{j['dset']}/{j['scene']}")
print("\nfailed:", failed or "none")

## 7. Results and the kill rule
Deltas are variant − base in dB (PSNR). "first" = frame 0 (before every training view: extrapolated); "interior" = between training views. `unseen` / `seen` = PSNR on pixels that no training view sees / that some training view sees.

In [ ]:
import math
import numpy as np

MODES = [m for m in VARIANTS.split(",") if m != "base"]
res = {}
for f in sorted(glob.glob(f"{RES_DRIVE}/*/*/uncertainty_splat.json")):
    dset, scene = f.split("/")[-3], f.split("/")[-2]
    res.setdefault(dset, {})[scene] = json.load(open(f))

def g(r, group, mode, key="psnr"):
    return r["summary"][group][mode][key]

def d(r, group, mode, key="psnr"):
    a, b = g(r, group, mode, key), g(r, group, "base", key)
    return None if a is None or b is None else a - b

def fmt(x, nd=2, sign=True):
    return "n/a" if x is None else (f"{x:+.{nd}f}" if sign else f"{x:.{nd}f}")

lines = [f"# C2 post-hoc test ({TAG}; sigma2 {SIGMA2}, kappa {KAPPA}, W0 {W0}, tau {TAU}, sigma_z {SIGMA_Z})", ""]
def out(s=""):
    print(s); lines.append(s)

for dset, scenes in res.items():
    out(f"## {dset}")
    out("")
    out("First test view (frame 0). PSNR base, then c2 − base on all / seen / never-seen pixels, and other variants on never-seen:")
    out("")
    out("| scene | seen % | base all / unseen | c2 Δall | c2 Δseen | c2 Δunseen | fade Δunseen | c2_det3 Δunseen | mip Δunseen | cull Δunseen |")
    out("|---|---|---|---|---|---|---|---|---|---|")
    for sc, r in scenes.items():
        first = [v for v in r["test"].values() if v["position"] == "start"]
        seen = 100 * np.mean([v["seen_fraction"] for v in first]) if first else float("nan")
        out(f"| {sc} | {seen:.1f} | {fmt(g(r, 'start', 'base'), sign=False)} / {fmt(g(r, 'start', 'base', 'psnr_unseen'), sign=False)} "
            f"| {fmt(d(r, 'start', 'c2'))} | {fmt(d(r, 'start', 'c2', 'psnr_seen'))} | {fmt(d(r, 'start', 'c2', 'psnr_unseen'))} | "
            + " | ".join(fmt(d(r, 'start', m, 'psnr_unseen')) if m in r['summary']['start'] else 'n/a'
                         for m in ('fade', 'c2_det3', 'mip', 'cull')) + " |")
    out("")
    out("All groups (PSNR change vs base) and training views (each with its own colour correction):")
    out("")
    out("| scene | Gaussians seen by 0 / 1 / ≥2 views | bound J P Jᵀ/σ² (≤ 1) | interior Δ c2 | all-views Δ c2 / fade / c2_det3 / mip / cull | train Δ c2 / fade / c2_det3 / mip / cull |")
    out("|---|---|---|---|---|---|")
    for sc, r in scenes.items():
        st = r["uncertainty_stats"]
        n = max(1, st["n_gaussians"])
        tr = r["summary"].get("train_psnr", {})
        trd = " / ".join(fmt(tr[m] - tr["base"]) if m in tr else "n/a" for m in ("c2", "fade", "c2_det3", "mip", "cull"))
        ald = " / ".join(fmt(d(r, "all", m)) if m in r["summary"]["all"] else "n/a" for m in ("c2", "fade", "c2_det3", "mip", "cull"))
        out(f"| {sc} | {100*st['views_0']/n:.1f} / {100*st['views_1']/n:.1f} / {100*st['views_2plus']/n:.1f} % "
            f"| {st['bound_check_max_ratio']:.3f} | {fmt(d(r, 'interior', 'c2'), 3)} | {ald} | {trd} |")
    out("")
    # dataset means, all test views (identity colour)
    for m in ["base"] + MODES:
        vals = [[g(r, "all", m, k) for r in scenes.values()] for k in ("psnr", "ssim", "lpips")]
        if all(None not in v for v in vals):
            out(f"- mean over {len(scenes)} scenes, all test views, `{m}`: "
                f"{np.mean(vals[0]):.2f} / {np.mean(vals[1]):.4f} / {np.mean(vals[2]):.4f}")
    out("")

    # ---- kill rule ----
    n = len(scenes)
    need = max(1, math.ceil(0.75 * n))
    p1 = []
    p3 = []
    p4 = []
    for sc, r in scenes.items():
        du, ds, da = d(r, "start", "c2", "psnr_unseen"), d(r, "start", "c2", "psnr_seen"), d(r, "start", "c2")
        ok1 = du is not None and ds is not None and da is not None and du >= 1.0 and ds >= -0.1 and da > 0
        p1.append(ok1)
        cu, fu = g(r, "start", "c2", "psnr_unseen"), (g(r, "start", "fade", "psnr_unseen") if "fade" in r["summary"]["start"] else None)
        p3.append(cu is not None and fu is not None and cu - fu > 0.1)
        ku = g(r, "start", "cull", "psnr_unseen") if "cull" in r["summary"]["start"] else None
        p4.append(cu is not None and ku is not None and cu - ku > 0.1)
    inter = [d(r, "interior", "c2") for r in scenes.values() if d(r, "interior", "c2") is not None]
    inter_mean = float(np.mean(inter)) if inter else None
    ok_inter = inter_mean is not None and abs(inter_mean) <= 0.05
    tr_ok = None
    if all("train_psnr" in r["summary"] and "mip" in r["summary"]["train_psnr"] for r in scenes.values()):
        loss_c2 = np.mean([r["summary"]["train_psnr"]["base"] - r["summary"]["train_psnr"]["c2"] for r in scenes.values()])
        loss_mip = np.mean([r["summary"]["train_psnr"]["base"] - r["summary"]["train_psnr"]["mip"] for r in scenes.values()])
        tr_ok = loss_c2 <= loss_mip + 0.02
    verdict = {
        f"(i) first view: Δunseen ≥ +1, Δseen ≥ −0.1, Δall > 0 in ≥ {need}/{n} scenes": f"{sum(p1)}/{n} -> {'PASS' if sum(p1) >= need else 'FAIL'}",
        "(i) interior views: mean Δ within ±0.05 dB": f"{fmt(inter_mean, 3)} -> {'PASS' if ok_inter else 'FAIL'}",
        "(ii) training-view loss of c2 ≤ post-hoc mip (+0.02 dB)": ("n/a" if tr_ok is None else
            f"c2 {loss_c2:+.3f} vs mip {loss_mip:+.3f} dB -> {'PASS' if tr_ok else 'FAIL'}"),
        f"(iii) c2 − fade on first-view never-seen > 0.1 dB in ≥ {need}/{n} scenes": f"{sum(p3)}/{n} -> {'PASS' if sum(p3) >= need else 'FAIL'}",
        f"(iv) c2 − cull on first-view never-seen > 0.1 dB in ≥ {need}/{n} scenes": f"{sum(p4)}/{n} -> {'PASS' if sum(p4) >= need else 'FAIL'}",
    }
    out(f"**Kill rule, {dset}:**")
    out("")
    for k, v in verdict.items():
        out(f"- {k}: {v}")
    alive = sum(p1) >= need and ok_inter and bool(tr_ok) and sum(p3) >= need and sum(p4) >= need
    out(f"- **C2 on {dset}: {'SURVIVES' if alive else 'KILLED (as pre-registered)'}**")
    out("")

with open(f"{RES_DRIVE}/summary.md", "w") as fh:
    fh.write("\n".join(lines) + "\n")
print(f"saved {RES_DRIVE}/summary.md")

## 8. Look at the first views
Each montage is one test view: GT | base | c2 | fade | c2_det3 | mip | cull (same order as `VARIANTS`). Where c2 works, content that no training view saw (usually a strip at the frame edge) should fade into water instead of showing stretched or floating splats, while the rest of the image stays unchanged.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

SHOW = "start"          # "start", "interior", "end" or "all"
labels = ["GT"] + VARIANTS.split(",")
for dset, scenes in res.items():
    for sc, r in scenes.items():
        for name, v in r["test"].items():
            if SHOW != "all" and v["position"] != SHOW:
                continue
            stem = os.path.splitext(name)[0]
            hits = glob.glob(f"{RES_DRIVE}/{dset}/{sc}/montages/{stem}_gt_*.png")
            if not hits:
                continue
            im = np.asarray(Image.open(hits[0]))
            w = im.shape[1] // len(labels)
            fig, axs = plt.subplots(1, len(labels), figsize=(4 * len(labels), 4 * im.shape[0] / w))
            for k, ax in enumerate(axs):
                ax.imshow(im[:, k * w:(k + 1) * w]); ax.axis("off")
                t = labels[k] if k == 0 else f"{labels[k]} {v[labels[k]]['psnr']:.2f} dB"
                ax.set_title(t, fontsize=10)
            fig.suptitle(f"{dset} / {sc} / {name} ({v['position']}, seen {100 * v['seen_fraction']:.1f}%)", fontsize=11)
            plt.tight_layout(); plt.show()